# CNN Image Classification with Keras — Complete Demonstration

**Dataset:** CIFAR-10  
**Framework:** TensorFlow / Keras

This notebook demonstrates the complete workflow:

1. Dataset loading and preprocessing
2. CNN architecture and tensor dimensions
3. Training curves
4. Sample predictions
5. Confusion matrix
6. Classification report
7. Multiclass ROC-AUC
8. Feature-map visualization
9. Grad-CAM visualization

The notebook is designed for an M.Tech./Ph.D. classroom demonstration.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 1. Load CIFAR-10

CIFAR-10 contains 10 image classes, each represented by a 32×32 RGB image.

Classes:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

y_train = y_train.flatten()
y_test = y_test.flatten()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Training images:", x_train.shape)
print("Test images    :", x_test.shape)


## 2. Preprocessing

Normalize pixels from `[0,255]` to `[0,1]`.


In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Pixel range:", x_train.min(), "to", x_train.max())


In [ ]:
plt.figure(figsize=(12, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()


## 3. Build the CNN

Architecture:

\[
32	imes32	imes3
ightarrow Conv(32)
ightarrow ReLU
ightarrow MaxPool
ightarrow Conv(64)
ightarrow ReLU
ightarrow MaxPool
ightarrow Flatten
ightarrow Dense(128)
ightarrow Dense(10)
\]

Tensor dimensions:

\[
32	imes32	imes3
ightarrow32	imes32	imes32
ightarrow16	imes16	imes32
ightarrow16	imes16	imes64
ightarrow8	imes8	imes64
ightarrow4096
ightarrow128
ightarrow10.
\]


In [ ]:
model = models.Sequential([
    layers.Input(shape=(32, 32, 3), name="input_image"),

    layers.Conv2D(32, (3, 3), padding="same",
                  activation="relu", name="conv1"),
    layers.MaxPooling2D((2, 2), name="pool1"),

    layers.Conv2D(64, (3, 3), padding="same",
                  activation="relu", name="conv2"),
    layers.MaxPooling2D((2, 2), name="pool2"),

    layers.Flatten(name="flatten"),
    layers.Dense(128, activation="relu", name="dense1"),
    layers.Dense(10, activation="softmax", name="predictions")
])

model.summary()


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)


## 4. Training Curves

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
test_loss, test_accuracy = model.evaluate(
    x_test, y_test, verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Test accuracy : {100*test_accuracy:.2f}%")


## 5. Predictions and Sample Visualization

In [ ]:
y_prob = model.predict(x_test, batch_size=128, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print("Probability matrix:", y_prob.shape)
print("Predicted labels:", y_pred.shape)


In [ ]:
plt.figure(figsize=(14, 7))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[i])

    true_name = class_names[y_test[i]]
    pred_name = class_names[y_pred[i]]
    confidence = y_prob[i, y_pred[i]]

    plt.title(
        f"True: {true_name}\nPred: {pred_name} ({confidence:.1%})",
        fontsize=9
    )
    plt.axis("off")

plt.tight_layout()
plt.show()


## 6. Confusion Matrix

Rows represent the **true class** and columns represent the **predicted class**.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 8))
plt.imshow(cm, interpolation="nearest")
plt.colorbar()

ticks = np.arange(len(class_names))
plt.xticks(ticks, class_names, rotation=45, ha="right")
plt.yticks(ticks, class_names)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("CIFAR-10 Confusion Matrix")

threshold = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i, str(cm[i, j]),
            ha="center", va="center",
            fontsize=8
        )

plt.tight_layout()
plt.show()


## 7. Classification Report

The report provides:

- **Precision**
- **Recall**
- **F1-score**
- **Support**


In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4
    )
)


## 8. Multiclass ROC-AUC

We use a **one-vs-rest** formulation. For each class, its probability is treated as the score for that class versus all other classes.

We report:

- AUC for every class
- Macro-average ROC-AUC
- ROC curves


In [ ]:
y_test_bin = label_binarize(
    y_test,
    classes=np.arange(len(class_names))
)

n_classes = len(class_names)

fpr, tpr, roc_auc = {}, {}, {}

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(
        y_test_bin[:, i],
        y_prob[:, i]
    )
    roc_auc[i] = auc(fpr[i], tpr[i])

macro_auc = roc_auc_score(
    y_test_bin,
    y_prob,
    multi_class="ovr",
    average="macro"
)

print(f"Macro-average ROC-AUC: {macro_auc:.4f}")

for i, name in enumerate(class_names):
    print(f"{name:12s}: {roc_auc[i]:.4f}")


In [ ]:
plt.figure(figsize=(10, 8))

for i in range(n_classes):
    plt.plot(
        fpr[i], tpr[i],
        label=f"{class_names[i]} (AUC={roc_auc[i]:.3f})"
    )

plt.plot([0, 1], [0, 1], "--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"Multiclass ROC Curves — Macro AUC = {macro_auc:.3f}")
plt.legend(loc="lower right", fontsize=8)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 9. Feature-Map Visualization

We create an intermediate model that outputs the activations of `conv1` and `conv2`.

For an image:

\[
Xightarrow Conv_1ightarrow ReLUightarrow Pool
ightarrow Conv_2.
\]

These feature maps show the learned intermediate representations.


In [ ]:
feature_model = tf.keras.Model(
    inputs=model.input,
    outputs=[
        model.get_layer("conv1").output,
        model.get_layer("conv2").output
    ]
)

sample_index = 0
sample_image = x_test[sample_index]

conv1_features, conv2_features = feature_model.predict(
    sample_image[np.newaxis, ...],
    verbose=0
)

conv1_features = conv1_features[0]
conv2_features = conv2_features[0]

print("Conv1 feature maps:", conv1_features.shape)
print("Conv2 feature maps:", conv2_features.shape)


In [ ]:
num_maps = min(16, conv1_features.shape[-1])

plt.figure(figsize=(12, 8))

for i in range(num_maps):
    plt.subplot(4, 4, i + 1)
    plt.imshow(conv1_features[:, :, i], cmap="viridis")
    plt.title(f"Conv1 filter {i+1}")
    plt.axis("off")

plt.suptitle(
    f"First-Layer Feature Maps — True class: "
    f"{class_names[y_test[sample_index]]}"
)

plt.tight_layout()
plt.show()


In [ ]:
num_maps = min(16, conv2_features.shape[-1])

plt.figure(figsize=(12, 8))

for i in range(num_maps):
    plt.subplot(4, 4, i + 1)
    plt.imshow(conv2_features[:, :, i], cmap="viridis")
    plt.title(f"Conv2 filter {i+1}")
    plt.axis("off")

plt.suptitle("Second-Layer Feature Maps")
plt.tight_layout()
plt.show()


# 10. Grad-CAM

For target class \(c\), Grad-CAM computes weights:

\[
lpha_k^c=
rac{1}{Z}\sum_{i,j}
rac{\partial y^c}{\partial A_{ij}^k}
\]

and then:

\[
oxed{
L_{\mathrm{Grad-CAM}}^c=
ReLU\left(\sum_klpha_k^cA^kight)
}
\]

Workflow:

1. Forward pass
2. Select target class
3. Compute gradients with respect to the selected convolutional layer
4. Global-average-pool the gradients
5. Weight the feature maps
6. Sum
7. Apply ReLU
8. Resize and overlay


In [ ]:
def make_gradcam_heatmap(
    img_array,
    model,
    last_conv_layer_name,
    pred_index=None
):
    grad_model = tf.keras.models.Model(
        model.inputs,
        [
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)

        if pred_index is None:
            pred_index = tf.argmax(predictions[0])

        class_score = predictions[:, pred_index]

    grads = tape.gradient(class_score, conv_outputs)

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(1, 2)
    )

    conv_outputs = conv_outputs[0]
    pooled_grads = pooled_grads[0]

    heatmap = tf.reduce_sum(
        conv_outputs * pooled_grads,
        axis=-1
    )

    heatmap = tf.maximum(heatmap, 0)

    max_value = tf.reduce_max(heatmap)

    heatmap = tf.where(
        max_value > 0,
        heatmap / max_value,
        heatmap
    )

    return heatmap.numpy()


In [ ]:
def display_gradcam(
    image,
    heatmap,
    title="Grad-CAM",
    alpha=0.45
):
    heatmap_resized = tf.image.resize(
        heatmap[..., np.newaxis],
        (image.shape[0], image.shape[1])
    ).numpy().squeeze()

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(image)
    plt.title("Original image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap_resized, cmap="jet")
    plt.title("Grad-CAM heatmap")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(image)
    plt.imshow(heatmap_resized, cmap="jet", alpha=alpha)
    plt.title(title)
    plt.axis("off")

    plt.tight_layout()
    plt.show()


## 11. Grad-CAM for the Predicted Class

In [ ]:
sample_index = 5

image = x_test[sample_index]
input_image = image[np.newaxis, ...]

predictions = model.predict(input_image, verbose=0)
predicted_class = int(np.argmax(predictions[0]))
confidence = float(predictions[0, predicted_class])

print("True class     :", class_names[y_test[sample_index]])
print("Predicted class:", class_names[predicted_class])
print("Confidence     :", f"{confidence:.2%}")

heatmap = make_gradcam_heatmap(
    input_image,
    model,
    last_conv_layer_name="conv2",
    pred_index=predicted_class
)

display_gradcam(
    image,
    heatmap,
    title=f"Predicted: {class_names[predicted_class]} ({confidence:.2%})"
)


## 12. Grad-CAM for the True Class

Grad-CAM is **class-specific**. We can explicitly request the attribution map for the true class rather than the predicted class.


In [ ]:
true_class = int(y_test[sample_index])

true_heatmap = make_gradcam_heatmap(
    input_image,
    model,
    last_conv_layer_name="conv2",
    pred_index=true_class
)

display_gradcam(
    image,
    true_heatmap,
    title=f"Grad-CAM for true class: {class_names[true_class]}"
)


# 13. Classroom Takeaways

### Evaluation
- Accuracy
- Confusion matrix
- Precision / Recall / F1
- One-vs-rest ROC-AUC

### Representation
- Intermediate feature maps

### Explainability
- Grad-CAM

### Important distinction

**Feature maps** show intermediate CNN activations.

**Grad-CAM** produces a class-specific spatial attribution map.

A Grad-CAM heatmap should be interpreted as an attribution/localization aid, not as proof of causal reasoning.
